<a href="https://colab.research.google.com/github/Shailesh2302/Tokenization/blob/main/Tokenization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## From text to Attention

Part 1 : Tokenization  

what does the model actually see?

In [ ]:
!pip install  tiktoken -q

In [2]:
import tiktoken
import numpy as np

#Load GPT-4's tokenizer
tokenizer = tiktoken.get_encoding("cl100k_base")
model = tiktoken.model
print(f"{tiktoken.list_encoding_names()}")

print(f"Vocabulary size : {tokenizer.n_vocab:,} tokens")


['gpt2', 'r50k_base', 'p50k_base', 'p50k_edit', 'cl100k_base', 'o200k_base', 'o200k_harmony']
Vocabulary size : 100,277 tokens


  # 1.1 Basic Tokenization


 Let's see how text gets converted to numbers.

In [ ]:
# Simple example

text = "This is a sample text."
tokens = tokenizer.encode(text)

print(f"Text: '{text}'")
print(f"Tokens: {tokens}")
print(f"Numbers of Tokens: {len(tokens)}")


print("\nToken breackdown")
for token in tokens:
  print(f" {token} -> '{tokenizer.decode([token])}'")


Text: 'This is a sample text.'
Tokens: [2028, 374, 264, 6205, 1495, 13]
Numbers of Tokens: 6

Token breackdown
 2028 -> 'This'
 374 -> ' is'
 264 -> ' a'
 6205 -> ' sample'
 1495 -> ' text'
 13 -> '.'


In [ ]:
# Let's try some more interesting examples
examples = {
    "Hello World",
    "don't",
    "I love AI",
    "supercalifragilisticexpialidocious",
    "The quick brown fox jumps over the lazy dog",
    "artificial intelligence",
    "🎸",
    "cafe",
    "    spaces    "
}


print("How different texts get tokenized:\n")
for text in examples:
  tokens = tokenizer.encode(text)
  print(f"Text: '{text}'")
  print(f"  -> {len(tokens)} tokens : {tokens}")

  # show  the pieces
  pieces = [tokenizer.decode([token]) for token in tokens]
  print(f"  -> pieces : {pieces}")
  print()


How different texts get tokenized:

Text: '    spaces    '
  -> 3 tokens : [262, 12908, 257]
  -> pieces : ['   ', ' spaces', '    ']

Text: '🎸'
  -> 3 tokens : [9468, 236, 116]
  -> pieces : ['�', '�', '�']

Text: 'don't'
  -> 2 tokens : [15357, 956]
  -> pieces : ['don', "'t"]

Text: 'The quick brown fox jumps over the lazy dog'
  -> 9 tokens : [791, 4062, 14198, 39935, 35308, 927, 279, 16053, 5679]
  -> pieces : ['The', ' quick', ' brown', ' fox', ' jumps', ' over', ' the', ' lazy', ' dog']

Text: 'Hello World'
  -> 2 tokens : [9906, 4435]
  -> pieces : ['Hello', ' World']

Text: 'artificial intelligence'
  -> 3 tokens : [472, 16895, 11478]
  -> pieces : ['art', 'ificial', ' intelligence']

Text: 'supercalifragilisticexpialidocious'
  -> 11 tokens : [13066, 3035, 278, 333, 4193, 321, 4633, 4683, 532, 307, 78287]
  -> pieces : ['sup', 'erc', 'al', 'if', 'rag', 'il', 'istic', 'exp', 'ial', 'id', 'ocious']

Text: 'cafe'
  -> 2 tokens : [936, 1897]
  -> pieces : ['ca', 'fe']

Text: 'I l

## 1.2 Why tokenization matter

# Token count effects:
  ### API cost (you pay per token)
  ### Context limit (GPT-4 has 128k token limit)
  ### Model behavior (some task break across token boundries)

In [3]:
# Compare token efffiency across different content types

test_cases = {
    "English prose" : "The quick brown fox jump over the lazy dog.",
    "Python code" : "def hello():\n   print('Hello, world')",
    "JSON" : '{"name" : "Alice", "age" : 30, "city" : "NYC"}',
    "Numbers" : "1223432435 32432344343 4356265767",
    "URL" : "https://colab.research.google.com/drive/1zCAhCowpZL2kvrH1WklrFjboQ_GxAKGv#scrollTo=aXnLW7DwspaR"
}

print("Token efficiency comparison\n")
for name, text in test_cases.items():
  tokens = tokenizer.encode(text)
  chars = len(text)
  ratio = chars / len(tokens)
  print(f"{name}")
  print(f"  {chars} chars -> {len(tokens)} tokens ({ratio:.2f} chars/token)")
  print()



Token efficiency comparison

English prose
  43 chars -> 10 tokens (4.30 chars/token)

Python code
  37 chars -> 10 tokens (3.70 chars/token)

JSON
  46 chars -> 22 tokens (2.09 chars/token)

Numbers
  33 chars -> 14 tokens (2.36 chars/token)

URL
  95 chars -> 49 tokens (1.94 chars/token)



**Key Insights**: Different content has different token efficiency
   

*   English prose : ~4 characters per token
*   Code/JSON : often less efficient (more token per character)
*   This effect your API costs!



## 1.3 The Token boundary Problem

Some tasks are hard because they require reasoning WITHIN tokens.

In [6]:
word = "strawberry"
tokens = tokenizer.encode(word)

print(f"Word : '{word}'")
print(f"Tokens : {tokens}")
print(f"Pieces : {[tokenizer.decode([t]) for t in tokens]}")
print()
print("the model sees these pieces, not individual lettera!")
print("Counting 'r's requires looking INSIDE tokens - that's hard>" )

Word : 'strawberry'
Tokens : [496, 675, 15717]
496
675
15717
